# Optimizing a Metasurface with Hardware Constraints

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jman4162/metasurface-py/blob/main/notebooks/02_optimization.ipynb)

This notebook demonstrates the **relax-then-quantize** optimization pipeline:
1. Optimize continuous phases using L-BFGS-B
2. Quantize to a discrete codebook (2-bit)
3. Refine via coordinate descent
4. Compare against naive analytical steering + quantization

In [ ]:
# !pip install metasurface-py

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from metasurface_py.geometry import RectangularLattice
from metasurface_py.elements import PhaseOnlyCell, DiscretePhaseSpace
from metasurface_py.surfaces import Metasurface
from metasurface_py.em import steering_phase, far_field_pattern
from metasurface_py.core.types import AngleGrid
from metasurface_py.optimize import MaxGainObjective, relax_then_quantize
from metasurface_py.plotting import (
    set_publication_style, plot_pattern_comparison, plot_convergence
)

set_publication_style(target="ieee_double")

## Setup: 12×12 Surface with 2-bit Quantization

In [ ]:
freq = 10e9
lam = 3e8 / freq
lattice = RectangularLattice(nx=12, ny=12, dx=lam/2, dy=lam/2)
cell = PhaseOnlyCell(state_space=DiscretePhaseSpace(num_bits=2))
surface = Metasurface(lattice=lattice, cell=cell)

target_theta = np.radians(30)
angles_opt = AngleGrid(
    theta=np.linspace(0.01, np.pi - 0.01, 90),
    phi=np.linspace(0, 2 * np.pi - 0.01, 36),
)
angles_plot = AngleGrid(
    theta=np.linspace(0.01, np.pi - 0.01, 360),
    phi=np.array([0.0]),
)
print(f"Surface: {lattice.num_elements} elements, 2-bit (4 states)")

## Baseline: Analytical Steering + Quantization

In [ ]:
phase_analytical = steering_phase(lattice, target_theta, 0.0, freq)
state_baseline = surface.set_state(phase_analytical).quantize()
pattern_baseline = far_field_pattern(surface, state_baseline, freq, angles_plot)

## Optimized: Relax-Then-Quantize Pipeline

The pipeline:
1. **Relax**: Optimize continuous phases with L-BFGS-B (ignoring discrete constraint)
2. **Quantize**: Project to nearest 2-bit codebook entry
3. **Refine**: Coordinate descent over codebook entries

In [ ]:
objective = MaxGainObjective(target_theta, 0.0, angles_opt)

result = relax_then_quantize(
    objective, surface, freq, angles_opt,
    continuous_method="L-BFGS-B",
    refine=True, maxiter=100, seed=42,
)

pattern_optimized = far_field_pattern(surface, result.state, freq, angles_plot)
print(f"Optimization runtime: {result.runtime_seconds:.2f}s")
print(f"Final objective: {result.objective_value:.2f}")

## Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))

plot_pattern_comparison(
    [(pattern_baseline, "Analytical + Quantize"),
     (pattern_optimized, "Optimized + Quantize")],
    cut_phi=0.0, ax=ax1,
)
ax1.set_xlim([0, 90])
ax1.set_ylim([-30, 0])
ax1.set_title("Pattern Comparison (2-bit)")

plot_convergence(
    {"Relax-then-quantize": result.convergence_history},
    ax=ax2, ylabel="Objective (neg. gain dBi)",
)
ax2.set_title("Convergence")

fig.tight_layout()
plt.show()